In [2]:
#!pip install tensorflow

In [3]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing

# Load dataset
housing = fetch_california_housing()

X = housing.data
y = housing.target

In [16]:
#X.shape

In [12]:
# import pandas as pd
# housing_df = pd.DataFrame(X, columns=housing.feature_names)

In [13]:
#housing_df

In [19]:
housing_df.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


In [22]:
X.shape[1]

8

In [23]:

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42 #random seed variable to ensure reproducibility
)

# Feature Scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build MLP Model
model = Sequential([
    Dense(
        256,
        activation='relu',
        kernel_initializer='he_normal',
        input_shape=(X_train.shape[1],)
    ),
    BatchNormalization(),
    Dropout(0.3),   #Deactivate 30% of neurons to prevent overfitting

    Dense(
        128,
        activation='relu',
        kernel_initializer='he_normal'
    ),
    BatchNormalization(),
    Dropout(0.3),

    Dense(
        64,
        activation='relu',
        kernel_initializer='he_normal'
    ),
    BatchNormalization(),

    Dense(1)  # Regression Output
])

# Adam Optimizer => Adaptive Moment Estimation, which adjusts learning rates during training for better convergence.
# Adam is a variation of gradient descent that computes adaptive learning rates for each parameter.

#AdamW is a variant of the Adam optimizer that incorporates weight decay regularization, which helps prevent overfitting by adding a penalty to the loss function based on the magnitude of the weights. This can lead to better generalization performance, especially in deep learning models. AdamW decouples the weight decay from the optimization steps, allowing for more effective regularization without affecting the adaptive learning rates of Adam.

optimizer = tf.keras.optimizers.Adam( 
    learning_rate=0.001
)

model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=[
        'mae',
        tf.keras.metrics.RootMeanSquaredError()
    ]
)

# Callbacks
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

checkpoint = ModelCheckpoint(
    "best_house_price_model.keras",
    save_best_only=True,
    monitor='val_loss'
)

# Train
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=200,  #Epochs refer to the number of complete passes through the training dataset during the training process. Each epoch allows the model to learn and adjust its parameters based on the training data, and multiple epochs are often necessary for the model to converge to a good solution.
    batch_size=32, #Batch size refers to the number of training samples that are processed together in one forward and backward pass during the training of a machine learning model. It determines how many samples are used to compute the gradient and update the model's parameters before moving on to the next batch. A smaller batch size can lead to more frequent updates and potentially faster convergence, while a larger batch size can provide more stable gradients but may require more memory and computational resources.
    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint
    ],
    verbose=1
)

# Evaluate
loss, mae, rmse = model.evaluate(X_test, y_test)

print(f"Test MAE : {mae:.4f}")
print(f"Test RMSE: {rmse:.4f}")

# Prediction
predictions = model.predict(X_test[:5])

print("Predictions:")
print(predictions)

c:\Users\prajn\miniconda3\envs\codebasics_dl\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/200
413/413 ━━━━━━━━━━━━━━━━━━━━ 16s 14ms/step - loss: 1.6971 - mae: 0.9631 - root_mean_squared_error: 1.3027 - val_loss: 0.5702 - val_mae: 0.5765 - val_root_mean_squared_error: 0.7551 - learning_rate: 0.0010
Epoch 2/200
413/413 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - loss: 0.6156 - mae: 0.5905 - root_mean_squared_error: 0.7846 - val_loss: 0.5302 - val_mae: 0.5209 - val_root_mean_squared_error: 0.7282 - learning_rate: 0.0010
Epoch 3/200
413/413 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - loss: 0.5320 - mae: 0.5411 - root_mean_squared_error: 0.7294 - val_loss: 0.4634 - val_mae: 0.4862 - val_root_mean_squared_error: 0.6808 - learning_rate: 0.0010
Epoch 4/200
413/413 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.4791 - mae: 0.5099 - root_mean_squared_error: 0.6922 - val_loss: 0.4325 - val_mae: 0.4836 - val_root_mean_squared_error: 0.6576 - learning_rate: 0.0010
Epoch 5/200
413/413 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.4577 - mae: 0.4969 - root_mean_squared_error: 0.6765 - val_loss: 0.43